In [1]:
import numpy as np
from rdkit import Chem
from pathlib import Path
import pandas as pd

In [3]:
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)

def boltzmann_weights(G: np.ndarray, T: float = 300.0) -> np.ndarray:
    k_B: float = Boltzmann * Avogadro * 0.000239005736
    delta_G = G - G.min()
    factors = np.exp(-delta_G / (k_B * T))
    return factors / factors.sum()

In [4]:
from ml_enhance import QuantumFPFileLoader

loader = QuantumFPFileLoader(Path("../data/QuantumFP/QFP_output"))
filelist = loader.list_output_files()

In [5]:
from collections.abc import Generator

def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for df in loader.stream_conformer_dataframe(file):
        df["gibbs_free_energy_300K"] = df["gibbs_free_energy"].map(lambda x: x[1][1])
        yield df

In [6]:
from data_preprocess import extract_atom_features, extract_bond_features, extract_mol_features

In [12]:
l = []
for file in filelist[:2]:
    for df in stream_conformer_df(file, loader): # filelist[1002]
        tdf = extract_mol_features(df)
        l.append(tdf)

combo_df = pd.concat(l, ignore_index=True)

In [13]:
combo_df #[["atom", "partial_charge"]]

,original_smiles,molecular_dipole_norm
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,1.513984
1,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,0.965182


In [ ]:
tdf.loc[0, "original_smiles"]

'[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]'

In [ ]:
mol = Chem.MolFromSmiles('[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]', sanitize=False)